In [92]:
import pandas as pd

from scipy.io import arff

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import hamming_loss, accuracy_score


import hvplot.pandas
from sklearn.metrics import confusion_matrix
import holoviews as hv

In [2]:
df_train = pd.read_csv('emotions/emotions_train.csv')

In [3]:
df_test = pd.read_csv('emotions/emotions_test.csv')

In [4]:
df_train.head()

,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,Mean_Acc1298_Mean_Mem40_MFCC_1,Mean_Acc1298_Mean_Mem40_MFCC_2,Mean_Acc1298_Mean_Mem40_MFCC_3,Mean_Acc1298_Mean_Mem40_MFCC_4,Mean_Acc1298_Mean_Mem40_MFCC_5,Mean_Acc1298_Mean_Mem40_MFCC_6,...,BH_HighLowRatio,BHSUM1,BHSUM2,BHSUM3,amazed-suprised,happy-pleased,relaxing-calm,quiet-still,sad-lonely,angry-aggresive
0,0.034741,0.089665,0.091225,-73.302422,6.215179,0.615074,2.037160,0.804065,1.301409,0.558576,...,2.0,0.245457,0.105065,0.405399,0,1,1,0,0,0
1,0.081374,0.272747,0.085733,-62.584437,3.183163,-0.218145,0.163038,0.620251,0.458514,0.041426,...,2.0,0.343547,0.276366,0.710924,1,0,0,0,0,1
2,0.110545,0.273567,0.084410,-65.235325,2.794964,0.639047,1.281297,0.757896,0.489412,0.627636,...,3.0,0.188693,0.045941,0.457372,0,1,0,0,0,1
3,0.042481,0.199281,0.093447,-80.305152,5.824409,0.648848,1.754870,1.495532,0.739909,0.809644,...,2.0,0.102839,0.241934,0.351009,0,0,1,0,0,0
4,0.074550,0.140880,0.079789,-93.697749,5.543229,1.064262,0.899152,0.890336,0.702328,0.490685,...,2.0,0.195196,0.310801,0.683817,0,0,0,1,0,0


In [5]:
df_test.head()

,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,Mean_Acc1298_Mean_Mem40_MFCC_1,Mean_Acc1298_Mean_Mem40_MFCC_2,Mean_Acc1298_Mean_Mem40_MFCC_3,Mean_Acc1298_Mean_Mem40_MFCC_4,Mean_Acc1298_Mean_Mem40_MFCC_5,Mean_Acc1298_Mean_Mem40_MFCC_6,...,BH_HighLowRatio,BHSUM1,BHSUM2,BHSUM3,amazed-suprised,happy-pleased,relaxing-calm,quiet-still,sad-lonely,angry-aggresive
0,0.036299,0.064986,0.082104,-72.710462,7.920220,0.134279,2.546373,0.671063,1.589821,0.576485,...,2.0,0.095982,0.520006,0.677943,0,0,1,1,1,0
1,0.161218,0.467820,0.096983,-71.298043,1.176349,1.871744,1.097346,0.641059,0.372797,0.991050,...,2.0,0.752210,0.576382,1.477141,1,0,0,0,0,1
2,0.115987,0.336879,0.079068,-64.570939,2.339044,0.714859,1.792451,0.611347,0.287022,0.772846,...,2.0,0.488375,0.004603,1.147727,0,0,0,0,1,0
3,0.086016,0.141845,0.081554,-81.141092,6.714252,-1.338896,1.326248,0.340032,1.290664,0.337209,...,2.0,0.430059,0.102757,1.276632,0,1,1,0,0,0
4,0.063232,0.140621,0.082097,-66.596131,5.594724,0.350716,1.023655,0.439544,0.855564,0.414784,...,2.0,1.788567,0.032760,3.076057,0,0,0,0,1,0


In [6]:
df_train.shape

(391, 78)

In [7]:
df_train.describe()

,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,Mean_Acc1298_Mean_Mem40_MFCC_1,Mean_Acc1298_Mean_Mem40_MFCC_2,Mean_Acc1298_Mean_Mem40_MFCC_3,Mean_Acc1298_Mean_Mem40_MFCC_4,Mean_Acc1298_Mean_Mem40_MFCC_5,Mean_Acc1298_Mean_Mem40_MFCC_6,...,BH_HighLowRatio,BHSUM1,BHSUM2,BHSUM3,amazed-suprised,happy-pleased,relaxing-calm,quiet-still,sad-lonely,angry-aggresive
count,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,...,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000
mean,0.069977,0.206432,0.086680,-73.024495,4.821225,0.596974,1.666577,0.642598,0.840467,0.509918,...,2.097187,0.442414,0.377433,0.995528,0.304348,0.273657,0.429668,0.227621,0.242967,0.335038
std,0.031177,0.122262,0.010021,7.430150,2.062840,1.027836,0.634509,0.454925,0.360613,0.302477,...,0.321483,0.362775,0.393236,0.690449,0.460720,0.446406,0.495663,0.419834,0.429425,0.472609
min,0.010201,0.038729,0.070932,-99.090802,0.051474,-2.235707,-0.604609,-1.143864,-0.237114,-0.444623,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.045485,0.115491,0.079328,-76.966920,3.288905,-0.095958,1.277283,0.387128,0.607217,0.327448,...,2.000000,0.186655,0.097626,0.509127,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.066195,0.176479,0.085116,-72.328740,4.384101,0.587896,1.706145,0.645530,0.818205,0.524997,...,2.000000,0.343781,0.242964,0.831004,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.087197,0.271220,0.092088,-68.548188,6.277184,1.259581,2.073187,0.925979,1.053052,0.696944,...,2.000000,0.563363,0.542230,1.271616,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000
max,0.188637,0.698277,0.159460,-56.297652,11.853471,3.910873,4.382370,2.046689,2.358098,1.484489,...,3.000000,1.795128,1.762948,3.422899,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 78 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Mean_Acc1298_Mean_Mem40_Centroid  391 non-null    float64
 1   Mean_Acc1298_Mean_Mem40_Rolloff   391 non-null    float64
 2   Mean_Acc1298_Mean_Mem40_Flux      391 non-null    float64
 3   Mean_Acc1298_Mean_Mem40_MFCC_0    391 non-null    float64
 4   Mean_Acc1298_Mean_Mem40_MFCC_1    391 non-null    float64
 5   Mean_Acc1298_Mean_Mem40_MFCC_2    391 non-null    float64
 6   Mean_Acc1298_Mean_Mem40_MFCC_3    391 non-null    float64
 7   Mean_Acc1298_Mean_Mem40_MFCC_4    391 non-null    float64
 8   Mean_Acc1298_Mean_Mem40_MFCC_5    391 non-null    float64
 9   Mean_Acc1298_Mean_Mem40_MFCC_6    391 non-null    float64
 10  Mean_Acc1298_Mean_Mem40_MFCC_7    391 non-null    float64
 11  Mean_Acc1298_Mean_Mem40_MFCC_8    391 non-null    float64
 12  Mean_Acc

## 0. Baseline - Logistic Regression

In [42]:
emo_cols = [
    'amazed-suprised',
    'happy-pleased',
    'relaxing-calm',
    'quiet-still',
    'sad-lonely',
    'angry-aggresive'
]

X_train = df_train.drop(columns=emo_cols)
y_train = df_train[emo_cols]

In [44]:
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', OneVsRestClassifier(
        LogisticRegression(max_iter=1000)
    ))
])

In [48]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [50]:
scoring = {
    'f1_macro': make_scorer(
        f1_score,
        average='macro',
        zero_division=0
    ),
    'f1_micro': make_scorer(
        f1_score,
        average='micro',
        zero_division=0
    )
}

cv_results = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

In [52]:
print(
    'Macro F1:',
    cv_results['test_f1_macro'].mean(),
    '+/-',
    cv_results['test_f1_macro'].std()
)

print(
    'Micro F1:',
    cv_results['test_f1_micro'].mean(),
    '+/-',
    cv_results['test_f1_micro'].std()
)

Macro F1: 0.597527834860028 +/- 0.04416856798657782
Micro F1: 0.6202961595852109 +/- 0.03215723034587658


In [70]:
report_df = pd.DataFrame(
    classification_report(
        y_train,
        y_pred_cv,
        target_names=emo_cols,
        zero_division=0,
        output_dict=True
    )
).T

report_df

,precision,recall,f1-score,support
amazed-suprised,0.612069,0.596639,0.604255,119.0
happy-pleased,0.402439,0.308411,0.349206,107.0
relaxing-calm,0.743902,0.726190,0.734940,168.0
quiet-still,0.776471,0.741573,0.758621,89.0
sad-lonely,0.628571,0.463158,0.533333,95.0
angry-aggresive,0.658915,0.648855,0.653846,131.0
micro avg,0.651703,0.593794,0.621402,709.0
macro avg,0.637061,0.580804,0.605700,709.0
weighted avg,0.643175,0.593794,0.615767,709.0
samples avg,0.613171,0.604859,0.577725,709.0


In [86]:
print(
    'Hamming Loss:',
    hamming_loss(y_train, y_pred_cv)
)

print(
    'Subset Accuracy:',
    accuracy_score(y_train, y_pred_cv)
)

Hamming Loss: 0.2186700767263427
Subset Accuracy: 0.22762148337595908


In [98]:
for i, emotion in enumerate(emo_cols):

    cm = confusion_matrix(
        y_train.iloc[:, i],
        y_pred_cv[:, i]
    )

    cm_df = pd.DataFrame(
        cm,
        index=['No', 'Yes'],
        columns=['No', 'Yes']
    )

    cm_long = (
        cm_df
        .rename_axis('Actual')
        .reset_index()
        .melt(
            id_vars='Actual',
            var_name='Predicted',
            value_name='Count'
        )
    )

    display(
        cm_long.hvplot.heatmap(
            x='Predicted',
            y='Actual',
            C='Count',
            title=f'Confusion Matrix — {emotion}',
            width=500,
            height=400,
            colorbar=True
        )
    )

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

## 1. Improved Logistic Regression